In [3]:
# CatBoost.py
# -------------------------------------------------------------
# Baseline + Tuned CatBoost for aurora intensity prediction (with visuals)
# Author: Group 10 (COMPSCI 760) — ported from RandomForest.py to CatBoost
# -------------------------------------------------------------
#   将模型替换为 CatBoost：
#   1) 读取 final-planb-24.csv（含 parse_dates=["time"]）
#   2) 目标列: keogram_mean；特征=除 time/keogram_* 外的所有列
#   3) 按年份进行时间感知切分: 训练<2018，验证=2018，测试=2019-2020
#   4) 使用 Pipeline(SimpleImputer->CatBoost) 避免 CV 泄漏
#   5) 两个 RandomizedSearchCV：
#        [A] boosting_type="Ordered"（时间序列更稳）
#        [B] boosting_type="Plain"
#      以 MAE 或 RMSE 作为 refit 选择最佳
#   6) 在 VAL/TEST 上评估，输出与保存对比图表和表格
# -------------------------------------------------------------

import os
import sys
import subprocess
import numpy as np
import pandas as pd


try:
    from catboost import CatBoostRegressor
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "catboost>=1.2"])
    from catboost import CatBoostRegressor

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    precision_score, recall_score, f1_score, precision_recall_curve
)
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# -------------------------------------------------------------
# 0. 全局开关
# -------------------------------------------------------------
PIPELINE_MODE = True


REFIT_METRIC = "mae"

FAST_MODE   = True
CPU_COUNT   = os.cpu_count() or 2
N_JOBS      = min(4, CPU_COUNT)        
N_SPLITS    = 2 if FAST_MODE else 3
N_ITER_A    = 20 if FAST_MODE else 40    
N_ITER_B    = 12 if FAST_MODE else 40    
SAVE_DIR_NAME = "figs_catboost"
SAVE_TABLES   = True


CLASSIFICATION_VIEW = False
CLASS_THRESH = 0.3
CLASS_THRESH_QUANTILE = None

# -------------------------------------------------------------
# 1. 读取数据集

import os


try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()  


CSV_PATH_OVERRIDE = r"D:\760\final-planb-24.csv"   


CANDIDATE_PATHS = [
    CSV_PATH_OVERRIDE,
    os.path.join(BASE_DIR, "final-planb-24.csv"),
    os.path.join(BASE_DIR, "datasets", "final-planb-24.csv"),
    os.path.join(BASE_DIR, "..", "datasets", "final-planb-24.csv"),
    "final-planb-24.csv",
]
CANDIDATE_PATHS = [p for p in CANDIDATE_PATHS if p]  # 去掉 None

CSV_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Could not find 'final-planb-24.csv'. "
        "Please set CSV_PATH_OVERRIDE (e.g., r'D:\\760\\final-planb-24.csv') "
        "or put the file next to this script / in ./datasets/."
    )

SAVE_DIR = os.path.join(BASE_DIR, SAVE_DIR_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

def _savefig(path, tight=True, dpi=150):
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=dpi)
    print(f"[Saved] {path}")


df = pd.read_csv(CSV_PATH, parse_dates=["time"])
print("Loaded:", CSV_PATH)
print("Shape before drop:", df.shape)
print("Time range:", df["time"].min(), "->", df["time"].max(), flush=True)

def _savefig(path, tight=True, dpi=150):
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=dpi)
    print(f"[Saved] {path}")

df = pd.read_csv(CSV_PATH, parse_dates=["time"])
print("Loaded:", CSV_PATH)
print("Shape before drop:", df.shape)
print("Time range:", df["time"].min(), "->", df["time"].max(), flush=True)

# -------------------------------------------------------------
# 2. 目标与特征
# -------------------------------------------------------------
TARGET_COL = "keogram_mean"

before = len(df)
df = df.dropna(subset=[TARGET_COL]).copy()
after = len(df)
print(f"Dropped rows with NaN target ({TARGET_COL}): {before - after}", flush=True)


drop_cols = ["time", "keogram_mean", "keogram_median", "keogram_max"]
features = [c for c in df.columns if c not in drop_cols]
assert TARGET_COL not in features, "Leakage: target column is in features!"

X_all = df[features]
y_all = df[TARGET_COL].values

# -------------------------------------------------------------
# 3. 时间感知切分
# -------------------------------------------------------------
train_idx = df[(df["time"] < "2018-01-01")].index
val_idx   = df[(df["time"] >= "2018-01-01") & (df["time"] < "2019-01-01")].index
test_idx  = df[(df["time"] >= "2019-01-01") & (df["time"] < "2021-01-01")].index

print("Split sizes:",
      "train =", len(train_idx),
      "val =", len(val_idx),
      "test =", len(test_idx), flush=True)

X_train_df = X_all.loc[train_idx]
X_val_df   = X_all.loc[val_idx]
X_test_df  = X_all.loc[test_idx]

y_train = y_all[df.index.get_indexer(train_idx)]
y_val   = y_all[df.index.get_indexer(val_idx)]
y_test  = y_all[df.index.get_indexer(test_idx)]

t_val  = df.loc[val_idx,  "time"].values
t_test = df.loc[test_idx, "time"].values

assert not np.isnan(y_train).any()
assert not np.isnan(y_val).any()
assert not np.isnan(y_test).any()

if not PIPELINE_MODE:
   
    print("PIPELINE_MODE=False: imputing outside the model (slight CV leakage).")
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train_df)
    X_val   = imputer.transform(X_val_df)
    X_test  = imputer.transform(X_test_df)
else:
    print("PIPELINE_MODE=True: imputer will be fitted inside CV folds via Pipeline.")
    X_train, X_val, X_test = X_train_df, X_val_df, X_test_df

# -------------------------------------------------------------
# 4. Scoring 与 CV
# -------------------------------------------------------------
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae":  "neg_mean_absolute_error",
}
assert REFIT_METRIC in scoring

# -------------------------------------------------------------
# 5. 构建 CatBoost 管道 + 随机搜索（A/B）
# -------------------------------------------------------------
def _make_estimator_cat(boosting_type: str):
    """
    构建 Pipeline(imputer -> CatBoostRegressor)
    - boosting_type: "Ordered" 或 "Plain"
    - 将 thread_count 设为 max(1, CPU_COUNT-1) 避免与外层并行冲突
    """
    base = CatBoostRegressor(
        loss_function="RMSE",           
        random_state=42,
        verbose=0,
        boosting_type=boosting_type,    
        thread_count=max(1, (CPU_COUNT or 2) - 1),
        allow_const_label=True
    )
    if PIPELINE_MODE:
        return Pipeline(steps=[
            ("imp", SimpleImputer(strategy="median")),
            ("cat", base),
        ])
    return base

def _prefix(param_name: str) -> str:
    return f"cat__{param_name}" if PIPELINE_MODE else param_name

# 公共超参空间（在 FAST_MODE 下收敛/速度折中）
common_space = {
    _prefix("iterations"):        [600, 900, 1200] if FAST_MODE else [800, 1200, 1600, 2000],
    _prefix("learning_rate"):     [0.02, 0.04, 0.06, 0.08, 0.10],
    _prefix("depth"):             [4, 6, 8, 10],
    _prefix("l2_leaf_reg"):       [1, 3, 5, 7, 10, 15],
    _prefix("subsample"):         [0.66, 0.8, 1.0],
    _prefix("bagging_temperature"):[0.0, 0.25, 0.5, 1.0, 5.0],  
    _prefix("min_data_in_leaf"):  [1, 5, 10, 20],
    _prefix("leaf_estimation_iterations"): [1, 4, 8],
    _prefix("random_strength"):   [0.0, 0.5, 1.0, 2.0],
   
    _prefix("grow_policy"):       ["SymmetricTree", "Depthwise"],
   
    _prefix("bootstrap_type"):    ["Bayesian", "MVS"],
}


search_ordered = RandomizedSearchCV(
    estimator=_make_estimator_cat(boosting_type="Ordered"),
    param_distributions=common_space,
    n_iter=N_ITER_A,
    cv=tscv,
    scoring=scoring,
    refit=REFIT_METRIC,
    n_jobs=N_JOBS,
    verbose=1,
    random_state=42,
)

print("\n[Search A] CatBoost (boosting_type='Ordered') ...")
search_ordered.fit(X_train, y_train)
print("  A: best params:", search_ordered.best_params_)
print(f"  A: best CV {REFIT_METRIC.upper()}: {-search_ordered.best_score_:.4f}")


search_plain = RandomizedSearchCV(
    estimator=_make_estimator_cat(boosting_type="Plain"),
    param_distributions=common_space,
    n_iter=N_ITER_B,
    cv=tscv,
    scoring=scoring,
    refit=REFIT_METRIC,
    n_jobs=N_JOBS,
    verbose=1,
    random_state=43,
)

print("\n[Search B] CatBoost (boosting_type='Plain') ...")
search_plain.fit(X_train, y_train)
print("  B: best params:", search_plain.best_params_)
print(f"  B: best CV {REFIT_METRIC.upper()}: {-search_plain.best_score_:.4f}")

# 选择总体最优
candidates = [search_ordered, search_plain]
best_search = max(candidates, key=lambda s: s.best_score_)
best_estimator = best_search.best_estimator_

print("\n[Selection] Choosing the overall better search by CV metric ...")
print("  Selected params:", best_search.best_params_)
print(f"  Selected CV {REFIT_METRIC.upper()}: {-best_search.best_score_:.4f}")

# -------------------------------------------------------------
# 6. 验证与测试评估
# -------------------------------------------------------------
def eval_and_print(split_name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    print(f"{split_name} -> MSE: {mse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}", flush=True)

print("\n=== Evaluation (Selected Best Model) ===", flush=True)
if PIPELINE_MODE:
    val_pred  = best_estimator.predict(X_val_df)
    test_pred = best_estimator.predict(X_test_df)
else:
    val_pred  = best_estimator.predict(X_val)
    test_pred = best_estimator.predict(X_test)

eval_and_print("VAL ", y_val,  val_pred)
eval_and_print("TEST", y_test, test_pred)

# -------------------------------------------------------------
# 7. Baselines（均值/中位数）
# -------------------------------------------------------------
def baseline_report(y_true, name="mean"):
    if name == "mean":
        yhat = np.full_like(y_true, fill_value=np.mean(y_true), dtype=float)
    elif name == "median":
        yhat = np.full_like(y_true, fill_value=np.median(y_true), dtype=float)
    else:
        raise ValueError("name must be 'mean' or 'median'")
    mse = mean_squared_error(y_true, yhat)
    mae = mean_absolute_error(y_true, yhat)
    rmse = np.sqrt(mse)
    print(f"Baseline ({name}) -> RMSE: {rmse:.4f}  MAE: {mae:.4f}")

print("\n=== Baselines ===")
baseline_report(y_val,  "mean")
baseline_report(y_val,  "median")
baseline_report(y_test, "mean")
baseline_report(y_test, "median")

# -------------------------------------------------------------
# 8. 结果表格（模型 vs 基线）
# -------------------------------------------------------------
def _metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

rows = []
rmse_m, mae_m = _metrics(y_val,  val_pred)
rows.append(("VAL",  "CatBoost-Model", rmse_m, mae_m))
rows.append(("VAL",  "Baseline-mean",
             np.sqrt(mean_squared_error(y_val,  np.full_like(y_val, y_val.mean(), dtype=float))),
             mean_absolute_error(y_val, np.full_like(y_val, y_val.mean(), dtype=float))))
rows.append(("VAL",  "Baseline-median",
             np.sqrt(mean_squared_error(y_val,  np.full_like(y_val, np.median(y_val), dtype=float))),
             mean_absolute_error(y_val, np.full_like(y_val, np.median(y_val), dtype=float))))

rmse_m, mae_m = _metrics(y_test, test_pred)
rows.append(("TEST", "CatBoost-Model", rmse_m, mae_m))
rows.append(("TEST", "Baseline-mean",
             np.sqrt(mean_squared_error(y_test, np.full_like(y_test, y_test.mean(), dtype=float))),
             mean_absolute_error(y_test, np.full_like(y_test, y_test.mean(), dtype=float))))
rows.append(("TEST", "Baseline-median",
             np.sqrt(mean_squared_error(y_test, np.full_like(y_test, np.median(y_test), dtype=float))),
             mean_absolute_error(y_test, np.full_like(y_test, np.median(y_test), dtype=float))))

comparison_df = pd.DataFrame(rows, columns=["Split", "Method", "RMSE", "MAE"])
print(comparison_df.to_string(index=False))

if SAVE_TABLES:
    comp_csv  = os.path.join(SAVE_DIR, "comparison_metrics.csv")
    comp_html = os.path.join(SAVE_DIR, "comparison_metrics.html")
    comparison_df.to_csv(comp_csv, index=False)
    comparison_df.to_html(comp_html, index=False)
    print(f"[Saved] {comp_csv}")
    print(f"[Saved] {comp_html}")

# -------------------------------------------------------------
# 9. 特征重要性（Top-15）与可视化
# -------------------------------------------------------------
def _extract_cat_model(estimator):
    """
    从 Pipeline 中取出 CatBoostRegressor
    """
    model = estimator
    if hasattr(estimator, "named_steps"):
        model = estimator.named_steps.get("cat", model)
    return model

def _safe_feature_importances(estimator):
    model = _extract_cat_model(estimator)
    if hasattr(model, "feature_importances_"):
        return np.array(model.feature_importances_, dtype=float)
    try:
        return np.array(model.get_feature_importance(), dtype=float)
    except Exception:
        return None

importances = _safe_feature_importances(best_estimator)
if importances is None:
    print("\n[Info] Feature importances not available.")
else:
    order = np.argsort(importances)[::-1][:15]
    top_feats = [(features[i], float(importances[i])) for i in order]
    print("\nTop-15 feature importances:")
    for name, score in top_feats:
        print(f"{name:20s}  {score:.4f}")

    plt.figure(figsize=(8, 5))
    names = [x[0] for x in top_feats]
    vals  = [x[1] for x in top_feats]
    ax = plt.gca()
    ax.barh(names[::-1], vals[::-1])
    ax.set_xlabel("Importance")
    ax.set_title("Top-15 Feature Importances (CatBoost)")
    _savefig(os.path.join(SAVE_DIR, "feature_importance_top15.png"))
    plt.close()

# -------------------------------------------------------------
# 10. 预测/残差/时间序列图
# -------------------------------------------------------------
def plot_pred_vs_actual(y_true, y_pred, split, save_path):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, s=10, alpha=0.6)
    minv = float(np.nanmin([y_true.min(), y_pred.min()]))
    maxv = float(np.nanmax([y_true.max(), y_pred.max()]))
    plt.plot([minv, maxv], [minv, maxv], linestyle='--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{split} — Predicted vs. Actual")
    _savefig(save_path)
    plt.close()

def plot_time_series(t, y_true, y_pred, split, save_path):
    plt.figure(figsize=(10, 4))
    plt.plot(t, y_true, linewidth=1, label="Actual")
    plt.plot(t, y_pred, linewidth=1, alpha=0.9, label="Predicted")
    plt.xlabel("Time")
    plt.ylabel(TARGET_COL)
    plt.title(f"{split} — Time Series: Actual vs. Predicted")
    plt.legend()
    _savefig(save_path)
    plt.close()

def plot_residuals_hist(y_true, y_pred, split, save_path):
    resid = y_pred - y_true
    plt.figure(figsize=(7, 4))
    plt.hist(resid, bins=40, alpha=0.8)
    plt.xlabel("Residual (Pred - Actual)")
    plt.ylabel("Count")
    plt.title(f"{split} — Residuals Histogram")
    _savefig(save_path)
    plt.close()

def plot_bar_metric_comparison(df_metrics, split, save_path):
    sub = df_metrics[df_metrics["Split"] == split].copy()
    labels = sub["Method"].tolist()
    x = np.arange(len(labels))
    width = 0.38

    fig = plt.figure(figsize=(9, 4.5))
    ax = plt.gca()
    ax.bar(x - width/2, sub["RMSE"].values, width, label="RMSE")
    ax.bar(x + width/2, sub["MAE"].values,  width, label="MAE")
    ax.set_xticks(x, labels, rotation=30, ha='right')
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=False))
    ax.set_title(f"{split} — Model vs Baselines")
    ax.legend()
    _savefig(save_path)
    plt.close()

plot_pred_vs_actual(y_val,  val_pred,  "VAL",  os.path.join(SAVE_DIR, "val_pred_vs_actual.png"))
plot_pred_vs_actual(y_test, test_pred, "TEST", os.path.join(SAVE_DIR, "test_pred_vs_actual.png"))

plot_time_series(t_val,  y_val,  val_pred,  "VAL",  os.path.join(SAVE_DIR, "val_timeseries.png"))
plot_time_series(t_test, y_test, test_pred, "TEST", os.path.join(SAVE_DIR, "test_timeseries.png"))

plot_residuals_hist(y_val,  val_pred,  "VAL",  os.path.join(SAVE_DIR, "val_residuals_hist.png"))
plot_residuals_hist(y_test, test_pred, "TEST", os.path.join(SAVE_DIR, "test_residuals_hist.png"))

plot_bar_metric_comparison(comparison_df, "VAL",  os.path.join(SAVE_DIR, "val_model_vs_baselines.png"))
plot_bar_metric_comparison(comparison_df, "TEST", os.path.join(SAVE_DIR, "test_model_vs_baselines.png"))

# -------------------------------------------------------------
# 11. 可选分类视图
# -------------------------------------------------------------
def _maybe_get_threshold(y_true):
    if CLASS_THRESH_QUANTILE is not None:
        return float(np.quantile(y_true, CLASS_THRESH_QUANTILE))
    return CLASS_THRESH

if CLASSIFICATION_VIEW:
    thr_val  = _maybe_get_threshold(y_val)
    thr_test = _maybe_get_threshold(y_test)

    y_val_bin  = (y_val  >= thr_val).astype(int)
    y_test_bin = (y_test >= thr_test).astype(int)

    val_pred_bin  = (val_pred  >= thr_val).astype(int)
    test_pred_bin = (test_pred >= thr_test).astype(int)

    def _cls_report(split, y_true_bin, y_pred_bin, scores, save_prefix):
        prec = precision_score(y_true_bin, y_pred_bin, zero_division=0)
        rec  = recall_score(y_true_bin, y_pred_bin, zero_division=0)
        f1   = f1_score(y_true_bin, y_pred_bin, zero_division=0)
        print(f"[{split} Classification] Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")

        p, r, _ = precision_recall_curve(y_true_bin, scores)
        plt.figure(figsize=(6, 5))
        plt.plot(r, p, linewidth=2)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"{split} — Precision-Recall Curve")
        _savefig(os.path.join(SAVE_DIR, f"{save_prefix}_pr_curve.png"))
        plt.close()

    _cls_report("VAL",  y_val_bin,  val_pred_bin,  val_pred,  "val")
    _cls_report("TEST", y_test_bin, test_pred_bin, test_pred, "test")
else:
    print("\n[Info] Classification view disabled (set CLASSIFICATION_VIEW=True to compute Recall/F1 & PR curves).")


Loaded: D:\760\final-planb-24.csv
Shape before drop: (78957, 23)
Time range: 2012-01-01 00:00:00 -> 2021-01-01 05:00:00
Loaded: D:\760\final-planb-24.csv
Shape before drop: (78957, 23)
Time range: 2012-01-01 00:00:00 -> 2021-01-01 05:00:00
Dropped rows with NaN target (keogram_mean): 68813
Split sizes: train = 7430 val = 849 test = 1865
PIPELINE_MODE=True: imputer will be fitted inside CV folds via Pipeline.

[Search A] CatBoost (boosting_type='Ordered') ...
Fitting 2 folds for each of 20 candidates, totalling 40 fits


C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
26 fits failed out of a total of 40.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
8 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 473, in fit
    self._final_estimator.fit(Xt, 

  A: best params: {'cat__subsample': 0.66, 'cat__random_strength': 1.0, 'cat__min_data_in_leaf': 10, 'cat__learning_rate': 0.04, 'cat__leaf_estimation_iterations': 1, 'cat__l2_leaf_reg': 10, 'cat__iterations': 600, 'cat__grow_policy': 'SymmetricTree', 'cat__depth': 8, 'cat__bootstrap_type': 'MVS', 'cat__bagging_temperature': 0.25}
  A: best CV MAE: 22.5058

[Search B] CatBoost (boosting_type='Plain') ...
Fitting 2 folds for each of 12 candidates, totalling 24 fits


C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
10 fits failed out of a total of 24.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 473, in fit
    self._final_estimator.fit(Xt,

  B: best params: {'cat__subsample': 1.0, 'cat__random_strength': 0.0, 'cat__min_data_in_leaf': 10, 'cat__learning_rate': 0.02, 'cat__leaf_estimation_iterations': 4, 'cat__l2_leaf_reg': 7, 'cat__iterations': 600, 'cat__grow_policy': 'SymmetricTree', 'cat__depth': 10, 'cat__bootstrap_type': 'MVS', 'cat__bagging_temperature': 0.25}
  B: best CV MAE: 22.4507

[Selection] Choosing the overall better search by CV metric ...
  Selected params: {'cat__subsample': 1.0, 'cat__random_strength': 0.0, 'cat__min_data_in_leaf': 10, 'cat__learning_rate': 0.02, 'cat__leaf_estimation_iterations': 4, 'cat__l2_leaf_reg': 7, 'cat__iterations': 600, 'cat__grow_policy': 'SymmetricTree', 'cat__depth': 10, 'cat__bootstrap_type': 'MVS', 'cat__bagging_temperature': 0.25}
  Selected CV MAE: 22.4507

=== Evaluation (Selected Best Model) ===
VAL  -> MSE: 574.7687  MAE: 16.9805  R2: 0.1555
TEST -> MSE: 509.7519  MAE: 16.5448  R2: 0.0761

=== Baselines ===
Baseline (mean) -> RMSE: 26.0891  MAE: 18.7218
Baseline (med

In [13]:
# CatBoost.py — Fast & Safe HPO (Ordered-only, fixed early-stopping)
# -------------------------------------------------------------
# - 仅 Ordered boosting：更稳且省时
# - 在训练集“按时间抽稀+限量”的代理集上做 RandomizedSearchCV
# - 用 HPO 最优超参新建管道，在完整训练集上拟合，并用验证集 early-stopping
# - 保持与 RandomForest.py 一致的数据方法与可视化
# -------------------------------------------------------------

import os, sys, subprocess, numpy as np, pandas as pd
try:
    from catboost import CatBoostRegressor
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "catboost>=1.2"])
    from catboost import CatBoostRegressor

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# ========================= Config ============================
REFIT_METRIC = "mae"      # "mae" 或 "rmse"
CPU_COUNT    = os.cpu_count() or 2
N_JOBS       = min(4, CPU_COUNT)
N_SPLITS     = 2          # TS 任务两折足够，提速

SAVE_DIR     = "figs_catboost"
TARGET_COL   = "keogram_mean"

# —— HPO 的“快”设置（可再调小以提速）——
MAX_HPO_ROWS        = 20000   # 代理集最多样本数（按时间顺序取前 N 条）
HPO_DOWNSAMPLE_STEP = 2       # 抽稀步长：每隔 step 取一条
N_ITER_HPO          = 20      # 随机搜索次数（可改为 12/16/8）
ITERATIONS_RANGE    = [300, 600, 900]  # 由早停自动截断

# ======================= Paths & IO ==========================
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

CSV_PATH_OVERRIDE = r"D:\760\final-planb-24.csv"  # 如不固定可改为 None
# CSV_PATH_OVERRIDE = None

CANDIDATE_PATHS = [
    CSV_PATH_OVERRIDE,
    os.path.join(BASE_DIR, "final-planb-24.csv"),
    os.path.join(BASE_DIR, "datasets", "final-planb-24.csv"),
    os.path.join(BASE_DIR, "..", "datasets", "final-planb-24.csv"),
    "final-planb-24.csv",
]
CANDIDATE_PATHS = [p for p in CANDIDATE_PATHS if p]
CSV_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError("Could not find 'final-planb-24.csv' — set CSV_PATH_OVERRIDE or place file in ./ or ./datasets/")

OUT_DIR = os.path.join(BASE_DIR, SAVE_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

def _savefig(path, tight=True, dpi=150):
    if tight: plt.tight_layout()
    plt.savefig(path, dpi=dpi); print("[Saved]", path)

# ======================= Load & Prep =========================
df = pd.read_csv(CSV_PATH, parse_dates=["time"])
print("Loaded:", CSV_PATH, "| shape:", df.shape, "| time:", df["time"].min(), "→", df["time"].max())

df = df.dropna(subset=[TARGET_COL]).copy()

# 仅保留数值特征，避免插补/模型对非数值报错；同时排除泄漏列
drop_cols = ["time", "keogram_mean", "keogram_median", "keogram_max"]
num_cols  = [c for c in df.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])]
if not num_cols:
    raise ValueError("No numeric features after filtering. Please check dataset columns.")
features = num_cols

X_all = df[features].astype(np.float32)
y_all = df[TARGET_COL].astype(np.float32).values

# ===================== Time-aware Splits =====================
train_idx = df[df["time"] <  "2018-01-01"].index
val_idx   = df[(df["time"] >= "2018-01-01") & (df["time"] < "2019-01-01")].index
test_idx  = df[(df["time"] >= "2019-01-01") & (df["time"] < "2021-01-01")].index
print("Split sizes:", "train=",len(train_idx), "val=",len(val_idx), "test=",len(test_idx))

X_train_df, X_val_df, X_test_df = X_all.loc[train_idx], X_all.loc[val_idx], X_all.loc[test_idx]
y_train = y_all[df.index.get_indexer(train_idx)]
y_val   = y_all[df.index.get_indexer(val_idx)]
y_test  = y_all[df.index.get_indexer(test_idx)]
t_val   = df.loc[val_idx,  "time"].values
t_test  = df.loc[test_idx, "time"].values

# ==================== HPO Proxy Subset =======================
order = X_train_df.index
if HPO_DOWNSAMPLE_STEP > 1:
    order = order[::HPO_DOWNSAMPLE_STEP]
if len(order) > MAX_HPO_ROWS:
    order = order[:MAX_HPO_ROWS]

X_train_hpo = X_train_df.loc[order]
y_train_hpo = y_train[np.isin(X_train_df.index.values, order)]
print(f"[HPO proxy] rows={len(X_train_hpo)} from {len(X_train_df)} | step={HPO_DOWNSAMPLE_STEP} | cap={MAX_HPO_ROWS}")

# ================== CV, Estimator, Search ====================
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
scoring = {"rmse":"neg_root_mean_squared_error", "mae":"neg_mean_absolute_error"}
assert REFIT_METRIC in scoring

def make_estimator():
    base = CatBoostRegressor(
        loss_function="RMSE",
        boosting_type="Ordered",                 # 稳定且更快
        random_state=42, verbose=0,
        thread_count=max(1, (CPU_COUNT or 2)-1),
        allow_const_label=True
    )
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("cat", base),
    ])

def p(k): return f"cat__{k}"

# 安全且精简的搜索空间（避免不兼容组合）
param_space = {
    p("iterations"):       ITERATIONS_RANGE,          # 训练轮数上限，早停会自动截断
    p("learning_rate"):    [0.03, 0.05, 0.07, 0.10],
    p("depth"):            [4, 6, 8],
    p("l2_leaf_reg"):      [3, 5, 7, 10],
    p("min_data_in_leaf"): [5, 10, 20],
    p("random_strength"):  [0.0, 0.5, 1.0],
}

search = RandomizedSearchCV(
    estimator=make_estimator(),
    param_distributions=param_space,
    n_iter=N_ITER_HPO,
    cv=tscv,
    scoring=scoring,
    refit=REFIT_METRIC,
    n_jobs=N_JOBS,
    verbose=1,
    random_state=2024,
    error_score=np.nan,   # 个别组合失败也不会中断搜索
)

print("\n[HPO] RandomizedSearch (Ordered, safe space) ...")
search.fit(X_train_hpo, y_train_hpo)
if not np.isfinite(search.best_score_):
    raise RuntimeError("HPO failed to produce a valid configuration. Try increasing N_ITER_HPO or proxy size.")
print("Best params:", search.best_params_)
print(f"Best CV {REFIT_METRIC.upper()}: {-search.best_score_:.5f}")

# ================= Final train on FULL train =================
# 不能在已拟合模型上改超参；新建管道并灌入 best_params
best_params = search.best_params_
final_model = make_estimator()
final_model.set_params(**best_params)

# 早停相关（必须在 fit 之前用 set_params 设置到模型超参上）
final_model.set_params(
    cat__use_best_model=True,
    cat__od_type="Iter",   # 也可 "IncToDec"
    cat__od_wait=60
)

# eval_set / verbose 作为 fit 的参数传进去
fit_params = {
    "cat__eval_set": (X_val_df, y_val),
    "cat__verbose": False
}
final_model.fit(X_train_df, y_train, **fit_params)

# ===================== Predict & Evaluate ====================
val_pred  = final_model.predict(X_val_df)
test_pred = final_model.predict(X_test_df)

def report(name, yt, yp):
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    r2   = r2_score(yt, yp)
    print(f"{name} -> RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")

print("\n=== Evaluation (final with early stopping) ===")
report("VAL ", y_val,  val_pred)
report("TEST", y_test, test_pred)

# ================== Baselines & Table ========================
rows=[]
def metrics(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred)), mean_absolute_error(y_true, y_pred)

rmse, mae = metrics(y_val, val_pred)
rows += [("VAL","CatBoost-Model",rmse,mae),
         ("VAL","Baseline-mean",
          np.sqrt(mean_squared_error(y_val, np.full_like(y_val, y_val.mean(), dtype=float))),
          mean_absolute_error(y_val, np.full_like(y_val, y_val.mean(), dtype=float))),
         ("VAL","Baseline-median",
          np.sqrt(mean_squared_error(y_val, np.full_like(y_val, np.median(y_val), dtype=float))),
          mean_absolute_error(y_val, np.full_like(y_val, np.median(y_val), dtype=float)))]

rmse, mae = metrics(y_test, test_pred)
rows += [("TEST","CatBoost-Model",rmse,mae),
         ("TEST","Baseline-mean",
          np.sqrt(mean_squared_error(y_test, np.full_like(y_test, y_test.mean(), dtype=float))),
          mean_absolute_error(y_test, np.full_like(y_test, y_test.mean(), dtype=float))),
         ("TEST","Baseline-median",
          np.sqrt(mean_squared_error(y_test, np.full_like(y_test, np.median(y_test), dtype=float))),
          mean_absolute_error(y_test, np.full_like(y_test, np.median(y_test), dtype=float)))]

comparison_df = pd.DataFrame(rows, columns=["Split","Method","RMSE","MAE"])
print("\n", comparison_df.to_string(index=False))
comparison_df.to_csv(os.path.join(OUT_DIR, "comparison_metrics.csv"), index=False)
comparison_df.to_html(os.path.join(OUT_DIR, "comparison_metrics.html"), index=False)

# =================== Importance & Plots ======================
cat = final_model.named_steps["cat"]
imps = cat.get_feature_importance()
order = np.argsort(imps)[::-1][:15]
names = [features[i] for i in order]
vals  = [float(imps[i]) for i in order]

print("\nTop-15 feature importances:")
for n, s in zip(names, vals):
    print(f"{n:20s} {s:.4f}")

plt.figure(figsize=(8,5))
plt.barh(names[::-1], vals[::-1])
plt.xlabel("Importance"); plt.title("Top-15 Feature Importances (CatBoost)")
_savefig(os.path.join(OUT_DIR, "feature_importance_top15.png")); plt.close()

def plot_pred_vs_actual(y_true, y_pred, split, path):
    plt.figure(figsize=(6,6))
    plt.scatter(y_true, y_pred, s=10, alpha=0.6)
    mn, mx = float(min(y_true.min(), y_pred.min())), float(max(y_true.max(), y_pred.max()))
    plt.plot([mn,mx],[mn,mx],'--')
    plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title(f"{split} — Pred vs Actual")
    _savefig(path); plt.close()

def plot_time_series(t, y_true, y_pred, split, path):
    plt.figure(figsize=(10,4))
    plt.plot(t, y_true, lw=1, label="Actual")
    plt.plot(t, y_pred, lw=1, label="Predicted")
    plt.xlabel("Time"); plt.ylabel(TARGET_COL); plt.title(f"{split} — Time Series"); plt.legend()
    _savefig(path); plt.close()

def plot_residuals_hist(y_true, y_pred, split, path):
    plt.figure(figsize=(7,4))
    plt.hist((y_pred - y_true), bins=40, alpha=0.8)
    plt.xlabel("Residual (Pred-Actual)"); plt.ylabel("Count"); plt.title(f"{split} — Residuals")
    _savefig(path); plt.close()

def plot_bar_metric_comparison(dfm, split, path):
    sub = dfm[dfm["Split"]==split].copy()
    labels=sub["Method"].tolist(); x=np.arange(len(labels)); w=0.38
    plt.figure(figsize=(9,4.5)); ax=plt.gca()
    ax.bar(x-w/2, sub["RMSE"].values, w, label="RMSE")
    ax.bar(x+w/2, sub["MAE"].values,  w, label="MAE")
    ax.set_xticks(x, labels, rotation=30, ha='right')
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.set_title(f"{split} — Model vs Baselines"); ax.legend()
    _savefig(path); plt.close()

plot_pred_vs_actual(y_val,  val_pred,  "VAL",  os.path.join(OUT_DIR, "val_pred_vs_actual.png"))
plot_pred_vs_actual(y_test, test_pred, "TEST", os.path.join(OUT_DIR, "test_pred_vs_actual.png"))
plot_time_series(t_val,  y_val,  val_pred,  "VAL",  os.path.join(OUT_DIR, "val_timeseries.png"))
plot_time_series(t_test, y_test, test_pred, "TEST", os.path.join(OUT_DIR, "test_timeseries.png"))
plot_residuals_hist(y_val,  val_pred,  "VAL",  os.path.join(OUT_DIR, "val_residuals_hist.png"))
plot_residuals_hist(y_test, test_pred, "TEST", os.path.join(OUT_DIR, "test_residuals_hist.png"))
plot_bar_metric_comparison(comparison_df, "VAL",  os.path.join(OUT_DIR, "val_model_vs_baselines.png"))
plot_bar_metric_comparison(comparison_df, "TEST", os.path.join(OUT_DIR, "test_model_vs_baselines.png"))


Loaded: D:\760\final-planb-24.csv | shape: (78957, 23) | time: 2012-01-01 00:00:00 → 2021-01-01 05:00:00
Split sizes: train= 7430 val= 849 test= 1865
[HPO proxy] rows=3715 from 7430 | step=2 | cap=20000

[HPO] RandomizedSearch (Ordered, safe space) ...
Fitting 2 folds for each of 20 candidates, totalling 40 fits
Best params: {'cat__random_strength': 1.0, 'cat__min_data_in_leaf': 20, 'cat__learning_rate': 0.03, 'cat__l2_leaf_reg': 3, 'cat__iterations': 600, 'cat__depth': 4}
Best CV MAE: 21.98562

=== Evaluation (final with early stopping) ===
VAL  -> RMSE: 23.3495  MAE: 16.8434  R2: 0.1990
TEST -> RMSE: 22.0696  MAE: 16.7621  R2: 0.1172

 Split          Method      RMSE       MAE
  VAL  CatBoost-Model 23.349532 16.843363
  VAL   Baseline-mean 26.089121 18.721823
  VAL Baseline-median 27.881224 16.791920
 TEST  CatBoost-Model 22.069650 16.762128
 TEST   Baseline-mean 23.488995 17.267667
 TEST Baseline-median 25.290931 15.416922

Top-15 feature importances:
ap                   37.9467
Kp

In [19]:
# CatBoost_LogWeighted.py
# -------------------------------------------------------------
# CatBoost + 自动 log1p 特征扩展（非负且偏度大的数值列）
#           + 极值样本加权（分位数/稳健z，可选）
#           + 时间感知切分 + Pipeline + RandomizedSearchCV（Ordered/Plain）
# -------------------------------------------------------------

import os
import sys
import subprocess
import numpy as np
import pandas as pd

# ---- CatBoost 安装检测 ----
try:
    from catboost import CatBoostRegressor
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "catboost>=1.2"])
    from catboost import CatBoostRegressor

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    precision_score, recall_score, f1_score, precision_recall_curve
)
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# ========================= 全局开关 ==========================
PIPELINE_MODE = True
REFIT_METRIC  = "mae"               # "mae" 或 "rmse"
FAST_MODE     = True

CPU_COUNT = os.cpu_count() or 2
N_JOBS    = min(4, CPU_COUNT)
N_SPLITS  = 2 if FAST_MODE else 3
N_ITER_A  = 20 if FAST_MODE else 40    # Ordered boosting
N_ITER_B  = 12 if FAST_MODE else 40    # Plain boosting

SAVE_DIR_NAME = "figs_catboost_log_weight"
SAVE_TABLES   = True

# ---- 分类视图（可选）----
CLASSIFICATION_VIEW   = False
CLASS_THRESH          = 0.3
CLASS_THRESH_QUANTILE = None

# ======================= Log 变换配置 ========================
SKEW_THRESHOLD    = 1.0              # 偏度阈值：>该值且非负 → 追加 <col>_log1p
ONLY_NONNEGATIVE  = True             # 仅对最小值>=0 的列 log1p
APPLY_LOG_TO_LIST = []               # 额外强制追加 log1p 的列名列表（可留空）

class AppendLog1p(BaseEstimator, TransformerMixin):
    """在训练折上选择列，并在 transform 时以追加列的形式输出 <col>_log1p。"""
    def __init__(self, skew_threshold=1.0, only_nonnegative=True,
                 force_cols=None, feature_names=None):
        # ！！！必须保存为同名属性，方便 sklearn.clone/get_params
        self.skew_threshold = skew_threshold
        self.only_nonnegative = only_nonnegative
        self.force_cols = force_cols
        self.feature_names = feature_names
        # 拟合后属性
        self.cols_to_log_ = []
        self.feature_names_ = None

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names if self.feature_names else None)

        numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
        skew = X[numeric_cols].skew(numeric_only=True).fillna(0.0)

        cols = []
        for col in numeric_cols:
            cond_skew = (skew.get(col, 0.0) > self.skew_threshold)
            cond_nonneg = (X[col].min(skipna=True) >= 0) if self.only_nonnegative else True
            if cond_skew and cond_nonneg:
                cols.append(col)

        force = self.force_cols or []
        for col in force:
            if col in X.columns and pd.api.types.is_numeric_dtype(X[col]):
                if (X[col].min(skipna=True) >= 0) or (not self.only_nonnegative):
                    if col not in cols:
                        cols.append(col)

        self.cols_to_log_ = sorted(cols)
        self.feature_names_ = list(X.columns)
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names if self.feature_names else None)
        X_new = X.copy()
        for col in self.cols_to_log_:
            X_new[col + "_log1p"] = np.log1p(X_new[col].astype(float))
        return X_new

    def get_feature_names_out(self, input_features=None):
        base = self.feature_names_ if self.feature_names_ is not None else (
            self.feature_names if self.feature_names is not None else (input_features or [])
        )
        return np.array(list(base) + [c + "_log1p" for c in self.cols_to_log_], dtype=object)

# ====================== Reweighting 配置 =====================
WEIGHTING_METHOD   = "quantile"  # "quantile" 或 "robust_z"
QUANTILE_STRENGTH  = 2.0         # 越大越强调尾部
Z_STRENGTH         = 2.0
WEIGHT_MIN         = 1.0
WEIGHT_MAX         = 5.0

def weights_quantile(y, strength=2.0, wmin=1.0, wmax=5.0):
    order = np.argsort(y, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.linspace(0, 1, len(y), endpoint=True)
    dist = np.abs(ranks - 0.5) / 0.5
    w = 1.0 + strength * dist
    return np.clip(w, wmin, wmax)

def weights_robust_z(y, strength=2.0, wmin=1.0, wmax=5.0):
    med = np.median(y)
    mad = np.median(np.abs(y - med)) + 1e-8
    z = np.abs(y - med) / mad
    w = 1.0 + strength * np.tanh(z / 2.0)
    return np.clip(w, wmin, wmax)

# ======================== 读取数据 ============================
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

CSV_PATH_OVERRIDE = r"D:\760\final-planb-24.csv"   

CANDIDATE_PATHS = [
    CSV_PATH_OVERRIDE,
    os.path.join(BASE_DIR, "final-planb-24.csv"),
    os.path.join(BASE_DIR, "datasets", "final-planb-24.csv"),
    os.path.join(BASE_DIR, "..", "datasets", "final-planb-24.csv"),
    "final-planb-24.csv",
]
CANDIDATE_PATHS = [p for p in CANDIDATE_PATHS if p]

CSV_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError("Could not find 'final-planb-24.csv'.")

SAVE_DIR = os.path.join(BASE_DIR, SAVE_DIR_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

def _savefig(path, tight=True, dpi=150):
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=dpi)
    print(f"[Saved] {path}")

df = pd.read_csv(CSV_PATH, parse_dates=["time"])
print("Loaded:", CSV_PATH)
print("Shape before drop:", df.shape)
print("Time range:", df["time"].min(), "->", df["time"].max(), flush=True)

# ======================== 目标与特征 =========================
TARGET_COL = "keogram_mean"

before = len(df)
df = df.dropna(subset=[TARGET_COL]).copy()
after = len(df)
print(f"Dropped rows with NaN target ({TARGET_COL}): {before - after}", flush=True)

drop_cols = ["time", "keogram_mean", "keogram_median", "keogram_max"]
features = [c for c in df.columns if c not in drop_cols]
assert TARGET_COL not in features, "Leakage: target column is in features!"

X_all = df[features]
y_all = df[TARGET_COL].values

# ======================= 时间感知切分 ========================
train_idx = df[(df["time"] < "2018-01-01")].index
val_idx   = df[(df["time"] >= "2018-01-01") & (df["time"] < "2019-01-01")].index
test_idx  = df[(df["time"] >= "2019-01-01") & (df["time"] < "2021-01-01")].index

print("Split sizes:",
      "train =", len(train_idx),
      "val =", len(val_idx),
      "test =", len(test_idx), flush=True)

X_train_df = X_all.loc[train_idx]
X_val_df   = X_all.loc[val_idx]
X_test_df  = X_all.loc[test_idx]

y_train = y_all[df.index.get_indexer(train_idx)]
y_val   = y_all[df.index.get_indexer(val_idx)]
y_test  = y_all[df.index.get_indexer(test_idx)]

t_val  = df.loc[val_idx,  "time"].values
t_test = df.loc[test_idx, "time"].values

assert not np.isnan(y_train).any()
assert not np.isnan(y_val).any()
assert not np.isnan(y_test).any()

print("PIPELINE_MODE=True: all preprocessing inside CV folds.")
X_train, X_val, X_test = X_train_df, X_val_df, X_test_df

# ========================= Scoring & CV ======================
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae":  "neg_mean_absolute_error",
}
assert REFIT_METRIC in scoring

# ========================= Reweighting =======================
if WEIGHTING_METHOD.lower() == "quantile":
    sample_weight_train = weights_quantile(y_train, strength=QUANTILE_STRENGTH,
                                           wmin=WEIGHT_MIN, wmax=WEIGHT_MAX)
    method_used = f"quantile(strength={QUANTILE_STRENGTH})"
else:
    sample_weight_train = weights_robust_z(y_train, strength=Z_STRENGTH,
                                           wmin=WEIGHT_MIN, wmax=WEIGHT_MAX)
    method_used = f"robust_z(strength={Z_STRENGTH})"

print(f"[Reweighting] {method_used} → range=({sample_weight_train.min():.3f}, {sample_weight_train.max():.3f}), mean={sample_weight_train.mean():.3f}")

# =============== 构建 CatBoost 管道 + 随机搜索 ================
def _make_estimator_cat(boosting_type: str):
    base = CatBoostRegressor(
        loss_function="RMSE",
        random_state=42,
        verbose=0,
        boosting_type=boosting_type,          # "Ordered" / "Plain"
        thread_count=max(1, (CPU_COUNT or 2) - 1),
        allow_const_label=True
    )
    return Pipeline(steps=[
        ("log", AppendLog1p(
            skew_threshold=SKEW_THRESHOLD,
            only_nonnegative=ONLY_NONNEGATIVE,
            force_cols=APPLY_LOG_TO_LIST,
            feature_names=features
        )),
        ("imp", SimpleImputer(strategy="median")),
        ("cat", base),
    ])

def _prefix(name: str) -> str:
    return f"cat__{name}"

common_space = {
    _prefix("iterations"):        [600, 900, 1200] if FAST_MODE else [800, 1200, 1600, 2000],
    _prefix("learning_rate"):     [0.02, 0.04, 0.06, 0.08, 0.10],
    _prefix("depth"):             [4, 6, 8, 10],
    _prefix("l2_leaf_reg"):       [1, 3, 5, 7, 10, 15],
    _prefix("subsample"):         [0.66, 0.8, 1.0],
    _prefix("bagging_temperature"):[0.0, 0.25, 0.5, 1.0, 5.0],
    _prefix("min_data_in_leaf"):  [1, 5, 10, 20],
    _prefix("leaf_estimation_iterations"): [1, 4, 8],
    _prefix("random_strength"):   [0.0, 0.5, 1.0, 2.0],
    _prefix("grow_policy"):       ["SymmetricTree", "Depthwise"],
    _prefix("bootstrap_type"):    ["Bayesian", "MVS"],
}

# A) Ordered boosting
search_ordered = RandomizedSearchCV(
    estimator=_make_estimator_cat("Ordered"),
    param_distributions=common_space,
    n_iter=N_ITER_A, cv=tscv, scoring=scoring, refit=REFIT_METRIC,
    n_jobs=N_JOBS, verbose=1, random_state=42
)
print("\n[Search A] CatBoost (Ordered) with log-transform + sample weights ...")
search_ordered.fit(X_train, y_train, **{"cat__sample_weight": sample_weight_train})
print("  A: best params:", search_ordered.best_params_)
print(f"  A: best CV {REFIT_METRIC.upper()}: {-search_ordered.best_score_:.4f}")

# B) Plain boosting
search_plain = RandomizedSearchCV(
    estimator=_make_estimator_cat("Plain"),
    param_distributions=common_space,
    n_iter=N_ITER_B, cv=tscv, scoring=scoring, refit=REFIT_METRIC,
    n_jobs=N_JOBS, verbose=1, random_state=43
)
print("\n[Search B] CatBoost (Plain) with log-transform + sample weights ...")
search_plain.fit(X_train, y_train, **{"cat__sample_weight": sample_weight_train})
print("  B: best params:", search_plain.best_params_)
print(f"  B: best CV {REFIT_METRIC.upper()}: {-search_plain.best_score_:.4f}")

# 选择总体最优
candidates     = [search_ordered, search_plain]
best_search    = max(candidates, key=lambda s: s.best_score_)
best_estimator = best_search.best_estimator_

print("\n[Selection] Choosing overall best ...")
print("  Selected params:", best_search.best_params_)
print(f"  Selected CV {REFIT_METRIC.upper()}: {-best_search.best_score_:.4f}")

# ======================== 评估验证/测试 ======================
def eval_and_print(split_name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    print(f"{split_name} -> MSE: {mse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}", flush=True)

print("\n=== Evaluation (Selected Best Model) ===", flush=True)
val_pred  = best_estimator.predict(X_val_df)
test_pred = best_estimator.predict(X_test_df)

eval_and_print("VAL ", y_val,  val_pred)
eval_and_print("TEST", y_test, test_pred)

# ======================== Baselines ==========================
def baseline_report(y_true, name="mean"):
    if name == "mean":
        yhat = np.full_like(y_true, fill_value=np.mean(y_true), dtype=float)
    elif name == "median":
        yhat = np.full_like(y_true, fill_value=np.median(y_true), dtype=float)
    else:
        raise ValueError("name must be 'mean' or 'median'")
    mse = mean_squared_error(y_true, yhat)
    mae = mean_absolute_error(y_true, yhat)
    rmse = np.sqrt(mse)
    print(f"Baseline ({name}) -> RMSE: {rmse:.4f}  MAE: {mae:.4f}")

print("\n=== Baselines ===")
baseline_report(y_val,  "mean")
baseline_report(y_val,  "median")
baseline_report(y_test, "mean")
baseline_report(y_test, "median")

# ===================== 结果表（保存） ========================
def _metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

rows = []
rmse_m, mae_m = _metrics(y_val,  val_pred)
rows.append(("VAL",  "CatBoost-Model (log+weights)", rmse_m, mae_m))
rows.append(("VAL",  "Baseline-mean",
             np.sqrt(mean_squared_error(y_val,  np.full_like(y_val, y_val.mean(), dtype=float))),
             mean_absolute_error(y_val, np.full_like(y_val, np.mean(y_val), dtype=float))))
rows.append(("VAL",  "Baseline-median",
             np.sqrt(mean_squared_error(y_val,  np.full_like(y_val, np.median(y_val), dtype=float))),
             mean_absolute_error(y_val, np.full_like(y_val, np.median(y_val), dtype=float))))

rmse_m, mae_m = _metrics(y_test, test_pred)
rows.append(("TEST", "CatBoost-Model (log+weights)", rmse_m, mae_m))
rows.append(("TEST", "Baseline-mean",
             np.sqrt(mean_squared_error(y_test, np.full_like(y_test, np.mean(y_test), dtype=float))),
             mean_absolute_error(y_test, np.full_like(y_test, np.mean(y_test), dtype=float))))
rows.append(("TEST", "Baseline-median",
             np.sqrt(mean_squared_error(y_test, np.full_like(y_test, np.median(y_test), dtype=float))),
             mean_absolute_error(y_test, np.full_like(y_test, np.median(y_test), dtype=float))))

comparison_df = pd.DataFrame(rows, columns=["Split", "Method", "RMSE", "MAE"])
print(comparison_df.to_string(index=False))

if SAVE_TABLES:
    comp_csv  = os.path.join(SAVE_DIR, "comparison_metrics.csv")
    comp_html = os.path.join(SAVE_DIR, "comparison_metrics.html")
    comparison_df.to_csv(comp_csv, index=False)
    comparison_df.to_html(comp_html, index=False)
    print(f"[Saved] {comp_csv}")
    print(f"[Saved] {comp_html}")

# ==================== 特征重要性（Top-15） ===================
def _extract_cat_model(estimator):
    model = estimator
    if hasattr(estimator, "named_steps"):
        model = estimator.named_steps.get("cat", model)
    return model

def _get_feature_names(estimator):
    if hasattr(estimator, "named_steps"):
        log_step = estimator.named_steps.get("log", None)
        if log_step is not None and hasattr(log_step, "get_feature_names_out"):
            return list(log_step.get_feature_names_out(features))
    return features

importances = None
try:
    model = _extract_cat_model(best_estimator)
    importances = np.array(model.get_feature_importance(), dtype=float)
except Exception:
    if hasattr(model, "feature_importances_"):
        importances = np.array(model.feature_importances_, dtype=float)

if importances is None:
    print("\n[Info] Feature importances not available.")
else:
    feat_names = _get_feature_names(best_estimator)
    topk = min(15, len(importances))
    order = np.argsort(importances)[::-1][:topk]
    top_feats = [(feat_names[i], float(importances[i])) for i in order]
    print("\nTop-15 feature importances:")
    for name, score in top_feats:
        print(f"{name:30s}  {score:.4f}")

    plt.figure(figsize=(9, 5))
    names = [x[0] for x in top_feats]
    vals  = [x[1] for x in top_feats]
    ax = plt.gca()
    ax.barh(names[::-1], vals[::-1])
    ax.set_xlabel("Importance")
    ax.set_title("Top Feature Importances (CatBoost + log)")
    _savefig(os.path.join(SAVE_DIR, "feature_importance_top.png"))
    plt.close()

# ======================= 可视化输出 ==========================
def plot_pred_vs_actual(y_true, y_pred, split, save_path):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, s=10, alpha=0.6)
    minv = float(np.nanmin([y_true.min(), y_pred.min()]))
    maxv = float(np.nanmax([y_true.max(), y_pred.max()]))
    plt.plot([minv, maxv], [minv, maxv], linestyle='--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{split} — Predicted vs. Actual")
    _savefig(save_path)
    plt.close()

def plot_time_series(t, y_true, y_pred, split, save_path):
    plt.figure(figsize=(10, 4))
    plt.plot(t, y_true, linewidth=1, label="Actual")
    plt.plot(t, y_pred, linewidth=1, alpha=0.9, label="Predicted")
    plt.xlabel("Time")
    plt.ylabel(TARGET_COL)
    plt.title(f"{split} — Time Series: Actual vs. Predicted")
    plt.legend()
    _savefig(save_path)
    plt.close()

def plot_residuals_hist(y_true, y_pred, split, save_path):
    resid = y_pred - y_true
    plt.figure(figsize=(7, 4))
    plt.hist(resid, bins=40, alpha=0.8)
    plt.xlabel("Residual (Pred - Actual)")
    plt.ylabel("Count")
    plt.title(f"{split} — Residuals Histogram")
    _savefig(save_path)
    plt.close()

def plot_bar_metric_comparison(df_metrics, split, save_path):
    sub = df_metrics[df_metrics["Split"] == split].copy()
    labels = sub["Method"].tolist()
    x = np.arange(len(labels))
    width = 0.38

    fig = plt.figure(figsize=(9, 4.5))
    ax = plt.gca()
    ax.bar(x - width/2, sub["RMSE"].values, width, label="RMSE")
    ax.bar(x + width/2, sub["MAE"].values,  width, label="MAE")
    ax.set_xticks(x, labels, rotation=30, ha='right')
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=False))
    ax.set_title(f"{split} — Model vs Baselines")
    ax.legend()
    _savefig(save_path)
    plt.close()

plot_pred_vs_actual(y_val,  val_pred,  "VAL",  os.path.join(SAVE_DIR, "val_pred_vs_actual.png"))
plot_pred_vs_actual(y_test, test_pred, "TEST", os.path.join(SAVE_DIR, "test_pred_vs_actual.png"))

plot_time_series(t_val,  y_val,  val_pred,  "VAL",  os.path.join(SAVE_DIR, "val_timeseries.png"))
plot_time_series(t_test, y_test, test_pred, "TEST", os.path.join(SAVE_DIR, "test_timeseries.png"))

plot_residuals_hist(y_val,  val_pred,  "VAL",  os.path.join(SAVE_DIR, "val_residuals_hist.png"))
plot_residuals_hist(y_test, test_pred, "TEST", os.path.join(SAVE_DIR, "test_residuals_hist.png"))

plot_bar_metric_comparison(comparison_df, "VAL",  os.path.join(SAVE_DIR, "val_model_vs_baselines.png"))
plot_bar_metric_comparison(comparison_df, "TEST", os.path.join(SAVE_DIR, "test_model_vs_baselines.png"))

# ===================== 可选分类视图 =========================
def _maybe_get_threshold(y_true):
    if CLASS_THRESH_QUANTILE is not None:
        return float(np.quantile(y_true, CLASS_THRESH_QUANTILE))
    return CLASS_THRESH

if CLASSIFICATION_VIEW:
    thr_val  = _maybe_get_threshold(y_val)
    thr_test = _maybe_get_threshold(y_test)

    y_val_bin  = (y_val  >= thr_val).astype(int)
    y_test_bin = (y_test >= thr_test).astype(int)

    val_pred_bin  = (val_pred  >= thr_val).astype(int)
    test_pred_bin = (test_pred >= thr_test).astype(int)

    def _cls_report(split, y_true_bin, y_pred_bin, scores, save_prefix):
        prec = precision_score(y_true_bin, y_pred_bin, zero_division=0)
        rec  = recall_score(y_true_bin, y_pred_bin, zero_division=0)
        f1   = f1_score(y_true_bin, y_pred_bin, zero_division=0)
        print(f"[{split} Classification] Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")

        p, r, _ = precision_recall_curve(y_true_bin, scores)
        plt.figure(figsize=(6, 5))
        plt.plot(r, p, linewidth=2)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"{split} — Precision-Recall Curve")
        _savefig(os.path.join(SAVE_DIR, f"{save_prefix}_pr_curve.png"))
        plt.close()

    _cls_report("VAL",  y_val_bin,  val_pred_bin,  val_pred,  "val")
    _cls_report("TEST", y_test_bin, test_pred_bin, test_pred, "test")
else:
    print("\n[Info] Classification view disabled (set CLASSIFICATION_VIEW=True to compute Recall/F1 & PR curves).")


Loaded: D:\760\final-planb-24.csv
Shape before drop: (78957, 23)
Time range: 2012-01-01 00:00:00 -> 2021-01-01 05:00:00
Dropped rows with NaN target (keogram_mean): 68813
Split sizes: train = 7430 val = 849 test = 1865
PIPELINE_MODE=True: all preprocessing inside CV folds.
[Reweighting] quantile(strength=2.0) → range=(1.000, 3.000), mean=2.000

[Search A] CatBoost (Ordered) with log-transform + sample weights ...
Fitting 2 folds for each of 20 candidates, totalling 40 fits


C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
26 fits failed out of a total of 40.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
8 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 473, in fit
    self._final_estimator.fit(Xt, 

  A: best params: {'cat__subsample': 0.66, 'cat__random_strength': 1.0, 'cat__min_data_in_leaf': 10, 'cat__learning_rate': 0.04, 'cat__leaf_estimation_iterations': 1, 'cat__l2_leaf_reg': 10, 'cat__iterations': 600, 'cat__grow_policy': 'SymmetricTree', 'cat__depth': 8, 'cat__bootstrap_type': 'MVS', 'cat__bagging_temperature': 0.25}
  A: best CV MAE: 23.0442

[Search B] CatBoost (Plain) with log-transform + sample weights ...
Fitting 2 folds for each of 12 candidates, totalling 24 fits


C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
10 fits failed out of a total of 24.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dell\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 473, in fit
    self._final_estimator.fit(Xt,

  B: best params: {'cat__subsample': 1.0, 'cat__random_strength': 0.0, 'cat__min_data_in_leaf': 10, 'cat__learning_rate': 0.02, 'cat__leaf_estimation_iterations': 4, 'cat__l2_leaf_reg': 7, 'cat__iterations': 600, 'cat__grow_policy': 'SymmetricTree', 'cat__depth': 10, 'cat__bootstrap_type': 'MVS', 'cat__bagging_temperature': 0.25}
  B: best CV MAE: 22.8458

[Selection] Choosing overall best ...
  Selected params: {'cat__subsample': 1.0, 'cat__random_strength': 0.0, 'cat__min_data_in_leaf': 10, 'cat__learning_rate': 0.02, 'cat__leaf_estimation_iterations': 4, 'cat__l2_leaf_reg': 7, 'cat__iterations': 600, 'cat__grow_policy': 'SymmetricTree', 'cat__depth': 10, 'cat__bootstrap_type': 'MVS', 'cat__bagging_temperature': 0.25}
  Selected CV MAE: 22.8458

=== Evaluation (Selected Best Model) ===
VAL  -> MSE: 614.0289  MAE: 17.4647  R2: 0.0979
TEST -> MSE: 537.8093  MAE: 16.9177  R2: 0.0252

=== Baselines ===
Baseline (mean) -> RMSE: 26.0891  MAE: 18.7218
Baseline (median) -> RMSE: 27.8812  MAE